# 02 - Evaluation Scoring Pipeline

**Evaluation pipeline using:**
- **Glider** (text-only LLM judge) on ports 8805, 8807
- **Llama-4-Scout** (VLM judge with image) on ports 8806, 8808

In [ ]:
# === Path Setup ===
import sys
from pathlib import Path

# Notebook paths - all ares notebooks are in artemis_final/notebooks/ares/
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent  # artemis_final/
ROOT_DIR = ARTEMIS_DIR.parent             # Which_VLM_Router/

# Add to sys.path for imports
for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 ARTEMIS_DIR: {ARTEMIS_DIR}")

# Cell 1: Setup
%load_ext autoreload
%autoreload 2


import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)-15s | %(message)s', datefmt='%H:%M:%S')
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)

In [ ]:
# Cell 2: Database Connection
from sqlalchemy import text
from ares.db.connection import get_engine

engine = get_engine()
print("Connected to DB")

# Apply migrations
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE vlm_evaluations ADD COLUMN IF NOT EXISTS judge_molmo_score FLOAT"))
    conn.execute(text("ALTER TABLE vlm_evaluations ADD COLUMN IF NOT EXISTS judge_molmo_rank_group INTEGER"))
    conn.execute(text("ALTER TABLE vlm_evaluations ADD COLUMN IF NOT EXISTS judge_molmo_raw TEXT"))
    conn.commit()
print("Migrations applied")

In [ ]:
# Cell 3: Configure Endpoints
from inference_engine.runners import OpenAIStyleRunner
from inference_engine.config import ModelEndpoint

# Your current vLLM setup:
# - Glider: ports 8805, 8807
# - Llama Scout: ports 8806, 8808

endpoints = [
    # Glider (text-only LLM judge)
    ModelEndpoint(
        name="glider-1",
        model_id="PatronusAI/glider",
        base_url="http://localhost:8805/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
    ModelEndpoint(
        name="glider-2",
        model_id="PatronusAI/glider",
        base_url="http://localhost:8807/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
    # Llama Scout (VLM judge with image)
    ModelEndpoint(
        name="vlm-judge-1",
        model_id="nvidia/Llama-4-Scout-17B-16E-Instruct-FP8",
        base_url="http://localhost:8806/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
    ModelEndpoint(
        name="vlm-judge-2",
        model_id="nvidia/Llama-4-Scout-17B-16E-Instruct-FP8",
        base_url="http://localhost:8808/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
]

runner = OpenAIStyleRunner(
    models=endpoints,
    request_timeout_s=180,
    max_workers=64
)

print(f"Configured {len(endpoints)} endpoints:")
for ep in endpoints:
    print(f"  - {ep.name}: {ep.base_url}")

In [ ]:
# Cell 4: Initialize Pipeline
from ares.evaluation.router_eval_pipeline import RouterEvalPipeline

pipeline = RouterEvalPipeline(
    engine=engine,
    runner=runner,
    glider_model_names=["glider-1", "glider-2"],
    vlm_judge_model_names=["vlm-judge-1", "vlm-judge-2"],
    tracker_path="eval_progress.json",
    use_glider=True,
    use_vlm_judge=True,
)

print("Pipeline initialized!")
print(f"Glider models: {pipeline.glider_model_names}")
print(f"VLM Judge models: {pipeline.vlm_judge_model_names}")

In [ ]:
# Cell 5: Run Evaluation
# This will:
# 1. Load samples per source_config
# 2. Compute static metrics (exact match, F1, etc.)
# 3. Compute confidence scores
# 4. Run Glider (text evaluator) on 2 GPUs
# 5. Run Llama Scout (VLM judge with image) on 2 GPUs
# 6. Write all results to vlm_evaluations table

pipeline.evaluate_all(
    batch_size=50,
    force=False,             # Set True to recompute all
    max_parallel_configs=2,  # Process 2 source_configs at a time
    split=None,              # Filter: 'train', 'val', 'test', or None
)

In [ ]:
# Cell 6: Verify Results
import pandas as pd

query = """
SELECT 
    r.model_name,
    COUNT(*) as total,
    ROUND(AVG(r.score_exact_match_normalized)::numeric, 3) as avg_em,
    ROUND(AVG(r.score_f1)::numeric, 3) as avg_f1,
    ROUND(AVG(e.glider_score)::numeric, 2) as avg_glider,
    ROUND(AVG(e.judge_molmo_score)::numeric, 2) as avg_vlm_judge,
    ROUND(AVG(e.judge_molmo_rank_group)::numeric, 2) as avg_rank
FROM vlm_responses r
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE r.ok = true
GROUP BY r.model_name
ORDER BY avg_vlm_judge DESC NULLS LAST
"""

results_df = pd.read_sql(query, engine)
print("Results by Model:")
results_df

In [ ]:
# Cell 7: Check Coverage
coverage_query = """
SELECT 
    COUNT(*) as total_responses,
    SUM(CASE WHEN r.score_exact_match IS NOT NULL THEN 1 ELSE 0 END) as has_static,
    SUM(CASE WHEN r.confidence_score IS NOT NULL THEN 1 ELSE 0 END) as has_confidence,
    SUM(CASE WHEN e.glider_score IS NOT NULL THEN 1 ELSE 0 END) as has_glider,
    SUM(CASE WHEN e.judge_molmo_score IS NOT NULL THEN 1 ELSE 0 END) as has_vlm_judge
FROM vlm_responses r
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE r.ok = true
"""

coverage_df = pd.read_sql(coverage_query, engine)
print("Metric Coverage:")
coverage_df.T